# Slide-level QC aggregation check

## Setup

In [2]:
IS_PUBLIC = False  # whether to use public cloud MLFlow
METADATA_CSV_URI = "..."
QC_METRICS_CSV_URI = "..."

In [3]:
import os

import mlflow
import pandas as pd


if IS_PUBLIC:
    os.environ["MLFLOW_TRACKING_USERNAME"] = "..."
    os.environ["MLFLOW_TRACKING_PASSWORD"] = "..."
    mlflow.set_tracking_uri("https://mlflow.rationai.cloud.e-infra.cz/")

metadata = pd.read_csv(mlflow.artifacts.download_artifacts(METADATA_CSV_URI))

/home/jovyan/prostate-cancer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
slides_to_remove = set()

## File Existance Check

In [ ]:
from pathlib import Path


df = pd.read_csv(mlflow.artifacts.download_artifacts(QC_METRICS_CSV_URI))

stems_original = metadata["slide_path"].map(lambda x: Path(x).stem)
stems_present = df["slide_name"]

set(stems_original) - set(stems_present)

In [6]:
slides_to_remove |= set(stems_original) - set(stems_present)

## Slide Level Mean & Medians

In [ ]:
df.columns

In [8]:
import matplotlib.pyplot as plt
import numpy as np


def icdf(df: pd.DataFrame, col: str) -> None:
    values = df[col].values
    t = np.linspace(0, 1, 100)
    proportions = [(values > thresh).sum() for thresh in t]

    # plot
    plt.figure()
    plt.plot(t, proportions)
    plt.xlabel("Threshold t")
    plt.ylabel(f"Proportion of {col} > t")
    plt.show()

We remove all slides with median QC coverage at least 0.3.

### Blur

In [ ]:
df_blur = df[["slide_name", "median_coverage(Piqe)"]].sort_values(
    by="median_coverage(Piqe)", ascending=False
)
df_blur.head()

In [ ]:
icdf(df, "median_coverage(Piqe)")

In [ ]:
for t in [0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]:
    print(len(df[df["median_coverage(Piqe)"] >= t]))

In [12]:
slides_to_remove |= set(df[df["median_coverage(Piqe)"] >= 0.3]["slide_name"])

### Residuals

In [ ]:
df_residual = df[
    ["slide_name", "median_coverage(ResidualArtifactsAndCoverage)"]
].sort_values(by="median_coverage(ResidualArtifactsAndCoverage)", ascending=False)
df_residual.head()

In [ ]:
icdf(df, "median_coverage(ResidualArtifactsAndCoverage)")

In [ ]:
for t in [0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]:
    print(len(df[df["median_coverage(ResidualArtifactsAndCoverage)"] >= t]))

In [16]:
slides_to_remove |= set(
    df[df["median_coverage(ResidualArtifactsAndCoverage)"] >= 0.3]["slide_name"]
)

### Folding

In [ ]:
df_residual = df[["slide_name", "median_coverage(FoldingFunction)"]].sort_values(
    by="median_coverage(FoldingFunction)", ascending=False
)
df_residual.head()

In [ ]:
icdf(df, "median_coverage(FoldingFunction)")

In [ ]:
for t in [0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]:
    print(len(df[df["median_coverage(FoldingFunction)"] >= t]))

In [20]:
slides_to_remove |= set(df[df["median_coverage(FoldingFunction)"] >= 0.3]["slide_name"])

## Finalze selection

In [ ]:
len(slides_to_remove)

Store the slides:

In [22]:
pd.DataFrame({"slide_stem": list(slides_to_remove)}).to_csv(
    "to_exclude_based_on_qc.csv", index=False
)

Check what files were removed:

In [23]:
df_excluded = df[df["slide_name"].isin(slides_to_remove)]

In [ ]:
df_excluded.head(10)

In [25]:
assert (
    len(
        df_excluded[
            (df_excluded["median_coverage(ResidualArtifactsAndCoverage)"] < 0.3)
            & (df_excluded["median_coverage(Piqe)"] < 0.3)
            & (df_excluded["median_coverage(FoldingFunction)"] < 0.3)
        ]
    )
    == 0
), "Excluded wrong slides"